Week 12 · Day 5 — Training a Transformer on a Toy Dataset
Why this matters

Now your Transformer actually learns a task.
We’ll use a synthetic sequence-to-sequence dataset (like copying or reversing numbers) to see the full training pipeline in action.

Theory Essentials

Toy task: input = number sequence, output = reversed sequence (or copy).

Training loop = embeddings → forward pass → cross-entropy loss → backprop → optimizer.

Masks: ensure autoregressive decoding.

Evaluation: accuracy on held-out sequences.

This builds intuition for real seq2seq tasks (translation, summarization).

In [2]:
# Setup
import torch, torch.nn as nn, torch.optim as optim
torch.manual_seed(42)

# Transformer model (from Day 4, simplified)
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=32, num_heads=2, d_ff=64, num_layers=2, max_len=20):
        super().__init__()
        self.src_emb = nn.Embedding(vocab_size, d_model)
        self.tgt_emb = nn.Embedding(vocab_size, d_model)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(position * div_term), torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

        self.enc_layers = nn.ModuleList([nn.TransformerEncoderLayer(d_model, num_heads, d_ff, batch_first=True) for _ in range(num_layers)])
        self.dec_layers = nn.ModuleList([nn.TransformerDecoderLayer(d_model, num_heads, d_ff, batch_first=True) for _ in range(num_layers)])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt, tgt_mask=None):
        src, tgt = self.src_emb(src) + self.pe[:, :src.size(1)], self.tgt_emb(tgt) + self.pe[:, :tgt.size(1)]
        memory = src
        for layer in self.enc_layers:
            memory = layer(memory)
        out = tgt
        for layer in self.dec_layers:
            out = layer(out, memory, tgt_mask=tgt_mask)
        return self.fc_out(out)

# Toy dataset: reverse numbers
def make_data(num_samples=2000, seq_len=10, vocab_size=20):
    X, Y = [], []
    for _ in range(num_samples):
        seq = torch.randint(1, vocab_size, (seq_len,))
        X.append(seq)
        Y.append(torch.flip(seq, dims=[0])) # reversed
    return torch.stack(X), torch.stack(Y)

vocab_size, seq_len = 20, 10
X, Y = make_data(2000, seq_len, vocab_size)
train_X, train_Y = X[:1600], Y[:1600]
test_X, test_Y = X[1600:], Y[1600:]

# Model + optimizer
model = Transformer(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Causal mask
def causal_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    return mask.masked_fill(mask==1, float("-inf"))

# Training loop
epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    tgt_inp = train_Y[:, :-1]      # teacher forcing (input all but last token)
    tgt_out = train_Y[:, 1:]       # expected shifted output
    mask = causal_mask(tgt_inp.size(1))

    logits = model(train_X, tgt_inp, tgt_mask=mask)
    loss = criterion(logits.reshape(-1, vocab_size), tgt_out.reshape(-1))
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

# Quick evaluation
model.eval()
with torch.no_grad():
    tgt_inp = test_Y[:, :-1]
    tgt_out = test_Y[:, 1:]
    mask = causal_mask(tgt_inp.size(1))
    logits = model(test_X, tgt_inp, tgt_mask=mask)
    preds = logits.argmax(-1)
    acc = (preds == tgt_out).float().mean().item()
    print("Test Accuracy:", acc)


Epoch 1/100, Loss: 3.1948
Epoch 2/100, Loss: 3.1485
Epoch 3/100, Loss: 3.1079
Epoch 4/100, Loss: 3.0761
Epoch 5/100, Loss: 3.0524
Epoch 6/100, Loss: 3.0314
Epoch 7/100, Loss: 3.0144
Epoch 8/100, Loss: 2.9973
Epoch 9/100, Loss: 2.9826
Epoch 10/100, Loss: 2.9685
Epoch 11/100, Loss: 2.9573
Epoch 12/100, Loss: 2.9493
Epoch 13/100, Loss: 2.9381
Epoch 14/100, Loss: 2.9291
Epoch 15/100, Loss: 2.9217
Epoch 16/100, Loss: 2.9141
Epoch 17/100, Loss: 2.9067
Epoch 18/100, Loss: 2.9005
Epoch 19/100, Loss: 2.8934
Epoch 20/100, Loss: 2.8876
Epoch 21/100, Loss: 2.8800
Epoch 22/100, Loss: 2.8726
Epoch 23/100, Loss: 2.8620
Epoch 24/100, Loss: 2.8559
Epoch 25/100, Loss: 2.8470
Epoch 26/100, Loss: 2.8380
Epoch 27/100, Loss: 2.8269
Epoch 28/100, Loss: 2.8198
Epoch 29/100, Loss: 2.8100
Epoch 30/100, Loss: 2.8023
Epoch 31/100, Loss: 2.7896
Epoch 32/100, Loss: 2.7799
Epoch 33/100, Loss: 2.7721
Epoch 34/100, Loss: 2.7642
Epoch 35/100, Loss: 2.7509
Epoch 36/100, Loss: 2.7453
Epoch 37/100, Loss: 2.7370
Epoch 38/1

1) Core (10–15 min)
Task: Run training with epochs=100. Compare accuracy to epochs=5.

5 epochs: 0.056

100 epochs: 0.674

2) Practice (10–15 min)
Task: Change seq_len from 10 → 15. Train again and check performance.

In [ ]:
X, Y = make_data(2000, seq_len=15, vocab_size=20)
# repeat training loop
train_X, train_Y = X[:1600], Y[:1600]
test_X, test_Y = X[1600:], Y[1600:]


epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    tgt_inp = train_Y[:, :-1]      # teacher forcing (input all but last token)
    tgt_out = train_Y[:, 1:]       # expected shifted output
    mask = causal_mask(tgt_inp.size(1))

    logits = model(train_X, tgt_inp, tgt_mask=mask)
    loss = criterion(logits.reshape(-1, vocab_size), tgt_out.reshape(-1))
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

# Quick evaluation
model.eval()
with torch.no_grad():
    tgt_inp = test_Y[:, :-1]
    tgt_out = test_Y[:, 1:]
    mask = causal_mask(tgt_inp.size(1))
    logits = model(test_X, tgt_inp, tgt_mask=mask)
    preds = logits.argmax(-1)
    acc = (preds == tgt_out).float().mean().item()
    print("Test Accuracy:", acc)



Epoch 1/100, Loss: 2.8897
Epoch 2/100, Loss: 2.8243
Epoch 3/100, Loss: 2.7759
Epoch 4/100, Loss: 2.7421
Epoch 5/100, Loss: 2.7186
Epoch 6/100, Loss: 2.7008
Epoch 7/100, Loss: 2.6862
Epoch 8/100, Loss: 2.6737
Epoch 9/100, Loss: 2.6615
Epoch 10/100, Loss: 2.6531
Epoch 11/100, Loss: 2.6460
Epoch 12/100, Loss: 2.6401
Epoch 13/100, Loss: 2.6378
Epoch 14/100, Loss: 2.6324
Epoch 15/100, Loss: 2.6303
Epoch 16/100, Loss: 2.6286
Epoch 17/100, Loss: 2.6231
Epoch 18/100, Loss: 2.6202
Epoch 19/100, Loss: 2.6159
Epoch 20/100, Loss: 2.6122
Epoch 21/100, Loss: 2.6068
Epoch 22/100, Loss: 2.6013
Epoch 23/100, Loss: 2.5954
Epoch 24/100, Loss: 2.5906
Epoch 25/100, Loss: 2.5828
Epoch 26/100, Loss: 2.5757
Epoch 27/100, Loss: 2.5675
Epoch 28/100, Loss: 2.5598
Epoch 29/100, Loss: 2.5503
Epoch 30/100, Loss: 2.5385
Epoch 31/100, Loss: 2.5302
Epoch 32/100, Loss: 2.5195
Epoch 33/100, Loss: 2.5090
Epoch 34/100, Loss: 2.4968
Epoch 35/100, Loss: 2.4865
Epoch 36/100, Loss: 2.4744
Epoch 37/100, Loss: 2.4640
Epoch 38/1

Smaller length gives higher accuracy

3) Stretch (optional, 10–15 min)
Task: Train the same model on a copy task instead of reverse.

In [6]:
def make_data(num_samples=2000, seq_len=10, vocab_size=20):
    X, Y = [], []
    for _ in range(num_samples):
        seq = torch.randint(1, vocab_size, (seq_len,))
        X.append(seq)
        Y.append(seq) 
    return torch.stack(X), torch.stack(Y)

vocab_size, seq_len = 15, 10
X, Y = make_data(2000, seq_len, vocab_size)
train_X, train_Y = X[:1600], Y[:1600]
test_X, test_Y = X[1600:], Y[1600:]

# Model + optimizer
model = Transformer(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Causal mask
def causal_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    return mask.masked_fill(mask==1, float("-inf"))

# Training loop
epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    tgt_inp = train_Y[:, :-1]      # teacher forcing (input all but last token)
    tgt_out = train_Y[:, 1:]       # expected shifted output
    mask = causal_mask(tgt_inp.size(1))

    logits = model(train_X, tgt_inp, tgt_mask=mask)
    loss = criterion(logits.reshape(-1, vocab_size), tgt_out.reshape(-1))
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

# Quick evaluation
model.eval()
with torch.no_grad():
    tgt_inp = test_Y[:, :-1]
    tgt_out = test_Y[:, 1:]
    mask = causal_mask(tgt_inp.size(1))
    logits = model(test_X, tgt_inp, tgt_mask=mask)
    preds = logits.argmax(-1)
    acc = (preds == tgt_out).float().mean().item()
    print("Test Accuracy:", acc)

Epoch 1/100, Loss: 2.8946
Epoch 2/100, Loss: 2.8341
Epoch 3/100, Loss: 2.7878
Epoch 4/100, Loss: 2.7539
Epoch 5/100, Loss: 2.7275
Epoch 6/100, Loss: 2.7035
Epoch 7/100, Loss: 2.6857
Epoch 8/100, Loss: 2.6698
Epoch 9/100, Loss: 2.6556
Epoch 10/100, Loss: 2.6413
Epoch 11/100, Loss: 2.6308
Epoch 12/100, Loss: 2.6230
Epoch 13/100, Loss: 2.6143
Epoch 14/100, Loss: 2.6098
Epoch 15/100, Loss: 2.6010
Epoch 16/100, Loss: 2.5957
Epoch 17/100, Loss: 2.5876
Epoch 18/100, Loss: 2.5798
Epoch 19/100, Loss: 2.5741
Epoch 20/100, Loss: 2.5644
Epoch 21/100, Loss: 2.5563
Epoch 22/100, Loss: 2.5469
Epoch 23/100, Loss: 2.5386
Epoch 24/100, Loss: 2.5275
Epoch 25/100, Loss: 2.5187
Epoch 26/100, Loss: 2.5092
Epoch 27/100, Loss: 2.5004
Epoch 28/100, Loss: 2.4896
Epoch 29/100, Loss: 2.4776
Epoch 30/100, Loss: 2.4700
Epoch 31/100, Loss: 2.4600
Epoch 32/100, Loss: 2.4472
Epoch 33/100, Loss: 2.4390
Epoch 34/100, Loss: 2.4312
Epoch 35/100, Loss: 2.4190
Epoch 36/100, Loss: 2.4117
Epoch 37/100, Loss: 2.4031
Epoch 38/1

Mini-Challenge (≤40 min)

Train on Reverse-Number Task with Variable Lengths.

Create sequences of length 5–15.

Train Transformer on this mixed dataset.

Evaluate accuracy separately for short vs long sequences.

Acceptance Criteria:

Model runs end-to-end with variable lengths.

Accuracy ≥70% on test set.

Short note: performance degrades with longer unseen lengths (why?).

In [7]:
# === Week 12 · Day 5 — Mini-Challenge: Variable-Length Reverse Task ===
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PAD = 0                       # reserve 0 as PAD (your vocab generates 1..vocab_size-1)
VOCAB_SIZE = 20               # same as before
MIN_L, MAX_L = 5, 15          # variable lengths
MAX_LEN_PAD = MAX_L           # we pad up to max length in dataset
BATCH_SIZE = 64
EPOCHS = 60                   # should comfortably reach ≥70% on test

# --- data ---
def make_variable_data(n=4000, min_len=5, max_len=15, vocab_size=20):
    X, Y, L = [], [], []
    for _ in range(n):
        L_i = torch.randint(min_len, max_len+1, (1,)).item()
        seq = torch.randint(1, vocab_size, (L_i,))
        rev = torch.flip(seq, dims=[0])

        # pad to MAX_LEN_PAD with PAD=0 (right padding)
        x_pad = torch.full((MAX_LEN_PAD,), PAD, dtype=torch.long)
        y_pad = torch.full((MAX_LEN_PAD,), PAD, dtype=torch.long)
        x_pad[:L_i] = seq
        y_pad[:L_i] = rev
        X.append(x_pad)
        Y.append(y_pad)
        L.append(L_i)
    return torch.stack(X), torch.stack(Y), torch.tensor(L)

Xv, Yv, Lv = make_variable_data(4000, MIN_L, MAX_L, VOCAB_SIZE)
# split
idx = int(0.8 * len(Xv))
train_X, train_Y, train_L = Xv[:idx], Yv[:idx], Lv[:idx]
test_X,  test_Y,  test_L  = Xv[idx:], Yv[idx:], Lv[idx:]

class RevVarLenDS(Dataset):
    def __init__(self, X, Y, L): self.X, self.Y, self.L = X, Y, L
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Y[i], self.L[i]

train_loader = DataLoader(RevVarLenDS(train_X, train_Y, train_L), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(RevVarLenDS(test_X,  test_Y,  test_L ), batch_size=BATCH_SIZE, shuffle=False)

# --- model/opt ---
model = Transformer(vocab_size=VOCAB_SIZE, d_model=64, num_heads=4, d_ff=128, num_layers=2, max_len=MAX_LEN_PAD).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)  # ignore PAD tokens in loss
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# causal mask for decoder length (max_len-1 because of shift)
def causal_mask(sz):
    m = torch.triu(torch.ones(sz, sz, device=device), diagonal=1)
    return m.masked_fill(m==1, float("-inf"))

# --- training ---
for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0.0
    for src, y_full, lengths in train_loader:
        src, y_full = src.to(device), y_full.to(device)

        # Teacher forcing: shift target
        tgt_inp = y_full[:, :-1]           # [B, T-1]
        tgt_out = y_full[:, 1:]            # [B, T-1]
        mask = causal_mask(tgt_inp.size(1))

        optimizer.zero_grad()
        logits = model(src, tgt_inp, tgt_mask=mask)  # [B, T-1, V]
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}/{EPOCHS}  |  train loss: {total_loss/len(train_loader):.4f}")

# --- evaluation (masked token accuracy), split short vs long ---
def token_accuracy(model, loader, split_name):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for src, y_full, lengths in loader:
            src, y_full, lengths = src.to(device), y_full.to(device), lengths.to(device)
            tgt_inp = y_full[:, :-1]
            tgt_out = y_full[:, 1:]
            mask = causal_mask(tgt_inp.size(1))
            logits = model(src, tgt_inp, tgt_mask=mask).argmax(-1)  # [B, T-1]

            # count only non-PAD positions in tgt_out
            nonpad = (tgt_out != PAD)
            correct += (logits.eq(tgt_out) & nonpad).sum().item()
            total += nonpad.sum().item()
    acc = correct / max(total, 1)
    print(f"{split_name} token accuracy: {acc*100:.2f}%")
    return acc

# overall
overall_acc = token_accuracy(model, test_loader, "Test (all)")

# short vs long splits
short_mask = test_L <= 10
long_mask  = test_L >= 11

short_loader = DataLoader(RevVarLenDS(test_X[short_mask], test_Y[short_mask], test_L[short_mask]),
                          batch_size=BATCH_SIZE, shuffle=False)
long_loader  = DataLoader(RevVarLenDS(test_X[long_mask],  test_Y[long_mask],  test_L[long_mask]),
                          batch_size=BATCH_SIZE, shuffle=False)

short_acc = token_accuracy(model, short_loader, "Test (short ≤10)")
long_acc  = token_accuracy(model, long_loader,  "Test (long 11–15)")

print("\nAcceptance check:")
print(f"- End-to-end variable lengths ✅")
print(f"- Test accuracy ≥70%?  {'✅' if overall_acc >= 0.70 else '❌'}")

# Quick note (why long unseen lengths degrade): positional patterns & exposure bias.
print("\nNote:")
print("Performance typically drops on longer sequences because the model saw fewer long examples; "
      "position encodings and the decoder's autoregressive errors compound over more steps (exposure bias).")


Epoch 001/60  |  train loss: 2.7755
Epoch 005/60  |  train loss: 0.6662
Epoch 010/60  |  train loss: 0.2431
Epoch 015/60  |  train loss: 0.1615
Epoch 020/60  |  train loss: 0.1296
Epoch 025/60  |  train loss: 0.0861
Epoch 030/60  |  train loss: 0.0654
Epoch 035/60  |  train loss: 0.0435
Epoch 040/60  |  train loss: 0.0488
Epoch 045/60  |  train loss: 0.0422
Epoch 050/60  |  train loss: 0.0381
Epoch 055/60  |  train loss: 0.0340
Epoch 060/60  |  train loss: 0.0400
Test (all) token accuracy: 99.99%
Test (short ≤10) token accuracy: 100.00%
Test (long 11–15) token accuracy: 99.98%

Acceptance check:
- End-to-end variable lengths ✅
- Test accuracy ≥70%?  ✅

Note:
Performance typically drops on longer sequences because the model saw fewer long examples; position encodings and the decoder's autoregressive errors compound over more steps (exposure bias).


Notes / Key Takeaways

Full training loop = data → forward → loss → backprop → optimizer.

Toy tasks (reverse/copy) build intuition for seq2seq learning.

Masks prevent cheating by looking ahead.

Performance depends on sequence length and training data.

Transformers handle parallel sequence modeling efficiently.

This framework generalizes to real tasks (translation, summarization).

Reflection

Why is teacher forcing used (feeding gold tokens to the decoder)?

How does task difficulty change as sequence length increases?

1. Why is teacher forcing used (feeding gold tokens to the decoder)?
Because the decoder predicts tokens one step at a time, small mistakes can quickly accumulate and derail the sequence. Teacher forcing stabilizes training by giving the model the true previous token instead of its own (possibly wrong) prediction, so it learns the mapping faster and converges more reliably.

2. How does task difficulty change as sequence length increases?
As sequences get longer, the model must maintain dependencies across more steps. Errors accumulate (exposure bias), attention has to cover more positions, and positional encodings reach unseen regions. All this makes longer unseen sequences harder, so accuracy usually drops as length grows.